In [7]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import accuracy_score

In [8]:
# -----------------------------
# Generate Student Dataset
# -----------------------------

np.random.seed(42)

data = {
    "study_hours": np.random.uniform(1,10,200),
    "attendance": np.random.uniform(50,100,200),
    "assignments": np.random.uniform(0,10,200)
}

df = pd.DataFrame(data)

df["pass"] = (df["study_hours"]*0.4 +
              df["attendance"]*0.3 +
              df["assignments"]*0.3 > 60).astype(int)

In [9]:
# Normalize
scaler = MinMaxScaler()
df.iloc[:,:-1] = scaler.fit_transform(df.iloc[:,:-1])

In [10]:
# -----------------------------
# Split clients
# -----------------------------

num_clients = 3
dataset = df.values
np.random.shuffle(dataset)

clients = np.array_split(dataset,num_clients)

In [11]:
# -----------------------------
# Model
# -----------------------------

num_features = dataset.shape[1]-1
global_W = np.zeros(num_features)
global_b = 0


def sigmoid(z):
    return 1/(1+np.exp(-z))


def local_train(data,W,b,lr=0.1,epochs=20):

    X = data[:,:-1]
    y = data[:,-1]

    W_local = W.copy()
    b_local = b

    for _ in range(epochs):

        pred = sigmoid(np.dot(X,W_local)+b_local)

        error = pred-y

        grad_W = np.dot(X.T,error)/len(y)
        grad_b = np.mean(error)

        W_local -= lr*grad_W
        b_local -= lr*grad_b

    return W_local,b_local,len(y)


def fedavg(updates):

    total = sum(size for _,_,size in updates)

    new_W = np.zeros_like(updates[0][0])
    new_b = 0

    for W,b,size in updates:

        weight = size/total

        new_W += weight*W
        new_b += weight*b

    return new_W,new_b

In [12]:
# -----------------------------
# Training
# -----------------------------

for r in range(5):

    updates=[]

    for client in clients:

        W,b,size = local_train(client,global_W,global_b)
        updates.append((W,b,size))

    global_W,global_b = fedavg(updates)

    print("Round",r+1,"completed")

Round 1 completed
Round 2 completed
Round 3 completed
Round 4 completed
Round 5 completed
